# PointChess Modern NNUE Training on Kaggle 2x NVIDIA T4 GPUs

This notebook trains a **Modern HalfKAv2** NNUE neural network for **PointChess** using:
- 200,000+ self-play games generated in parallel with native C++ engine.
- PyTorch Distributed Data Parallel (DDP) across **2x NVIDIA T4 GPUs**.
- **SCReL** (Squared Clipped ReLU) activation and **AMP fp16** mixed precision.
- Exports custom **`.pchess`** (`pointchess.pchess`) and **`.pnet`** (`nnue.pnet`) formats.

In [ ]:
# Step 1: Verify Kaggle 2x NVIDIA T4 GPUs
!nvidia-smi
import torch
num_gpus = torch.cuda.device_count()
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs detected: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Step 2: Unpack PointChess Source Code
import os, base64
if not os.path.exists("src"):
    print("Unpacking embedded PointChess source...")
    b64_src = "..."
    with open("pointchess_src.tar.gz", "wb") as f:
        f.write(base64.b64decode(b64_src))
    !tar -xzf pointchess_src.tar.gz
print("PointChess directory contents:")
!ls -la src/ training/

In [ ]:
# Step 3: Compile PointChess C++ Engine with CMake
!mkdir -p build && cd build && cmake -DCMAKE_BUILD_TYPE=Release .. && make -j$(nproc)
!./build/pointchess bench 64 1 8

In [ ]:
# Step 4: Generate 200,000 Self-Play Games via Native Multi-Threaded Engine
# Runs at >1,000 games/sec across CPU cores
!./build/pointchess genselfplay 200000 $(nproc) 2 /kaggle/working/selfplay_200k.txt
!ls -lh /kaggle/working/selfplay_200k.txt && wc -l /kaggle/working/selfplay_200k.txt

In [ ]:
# Step 5: Multi-GPU PyTorch DDP Training on Kaggle 2x T4 GPUs
!cd training && torchrun --nproc_per_node=2 train.py \
    --data_file /kaggle/working/selfplay_200k.txt \
    --epochs 25 \
    --batch_size 1024 \
    --lr 1e-3 \
    --activation screl \
    --amp \
    --output_dir /kaggle/working/checkpoints

In [ ]:
# Step 6: Export Custom .pchess and .pnet Network Files
!cd training && python3 export.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pt \
    --output_pchess /kaggle/working/pointchess.pchess \
    --output_pnet /kaggle/working/nnue.pnet

!ls -lh /kaggle/working/pointchess.pchess /kaggle/working/nnue.pnet

In [ ]:
# Step 7: Verify PointChess Engine Running with Newly Trained .pchess Network
!echo -e "uci\nsetoption name EvalFile value /kaggle/working/pointchess.pchess\nsetoption name Use NNUE value true\nposition startpos\ngo depth 10\nquit" | ./build/pointchess